In [ ]:
import pandas as pd
import numpy as np
import sqlite3

df_synop = pd.read_csv(
    "synop_2025.csv.gz",
    sep=";",
    low_memory=False
)
df_stations = pd.read_csv(
    "observations-liste-stations-synop-omm-20250710.csv",
    sep=";"
)

In [ ]:
df_synop.head()

In [ ]:
df_synop.info()

In [ ]:
df_synop.columns


In [ ]:
df_synop.shape

In [ ]:
df_synop.describe()

In [ ]:
df_synop.isnull().sum()

In [ ]:
df_synop.duplicated().sum()

In [ ]:
df_synop = df_synop.drop_duplicates()

print("Nombre de doublons :", df_synop.duplicated().sum())

print(df_synop.shape)

In [ ]:
colonnes_vides = [
    "sw",
    "tw",
    "phenspe1",
    "phenspe2",
    "phenspe3",
    "phenspe4"
]

df_synop = df_synop.drop(columns=colonnes_vides)

In [ ]:
df_synop.shape

In [ ]:
df_stations.head()

In [ ]:
df_stations.shape

In [ ]:
df_stations.columns

In [ ]:
df_stations

In [ ]:
stations_occitanie = [7535,7558,7621,7627,7630,7643,7747]

df_region = df_synop[
    df_synop["geo_id_wmo"].isin(stations_occitanie)
].copy()

In [ ]:
df_region.shape

In [ ]:
df_region[
    ["geo_id_wmo", "name"]
].drop_duplicates().sort_values("geo_id_wmo")

In [ ]:
df_region["name"].value_counts()

In [ ]:
df_region["validity_time"] = pd.to_datetime(
    df_region["validity_time"],
    errors="coerce",
    utc=True
)

In [ ]:
print("Première date :", df_region["validity_time"].min())
print("Dernière date :", df_region["validity_time"].max())

In [ ]:
df_region = df_region.dropna(
    subset=["validity_time"]
).copy()

In [ ]:
df_region = df_region.sort_values(
    ["geo_id_wmo", "validity_time"]
).reset_index(drop=True)

In [ ]:
colonnes_utiles = [
    "validity_time",
    "geo_id_wmo",
    "name",
    "lat",
    "lon",
    "pmer",
    "tend",
    "dd",
    "ff",
    "t",
    "td",
    "u",
    "vv",
    "n",
    "rr1"
]

In [ ]:
df_prepare = df_region[colonnes_utiles].copy()

In [ ]:
df_prepare["validity_time"] = pd.to_datetime(
    df_prepare["validity_time"],
    errors="coerce",
    utc=True)

df_prepare = df_prepare.dropna(
    subset=["validity_time"]).copy()

df_prepare = df_prepare.sort_values(
    ["geo_id_wmo", "validity_time"]).reset_index(drop=True)

print(df_prepare["validity_time"].dtype)
print(df_prepare.shape)

In [ ]:
df_prepare["date_suivante"] = (
    df_prepare
    .groupby("geo_id_wmo")["validity_time"]
    .shift(-1))

df_prepare["rr1_suivant"] = (
    df_prepare
    .groupby("geo_id_wmo")["rr1"]
    .shift(-1))


df_prepare["delai_heures"] = (df_prepare["date_suivante"] - df_prepare["validity_time"]).dt.total_seconds() / 3600

In [ ]:
df_prepare["delai_heures"].value_counts().sort_index()

In [ ]:
df_prepare = df_prepare[
    (df_prepare["delai_heures"] == 3)
    & (df_prepare["rr1_suivant"].notna())].copy()

In [ ]:
df_prepare["pluie_prochaine_3h"] = (
    df_prepare["rr1_suivant"] > 0).astype(int)

In [ ]:
df_prepare = df_prepare.drop(
    columns=[
        "date_suivante",
        "rr1_suivant",
        "delai_heures"
    ])

print(df_prepare.shape)

In [ ]:
df_prepare["temperature_c"] = (df_prepare["t"] - 273.15)

df_prepare["point_rosee_c"] = (df_prepare["td"] - 273.15)

df_prepare["pression_hpa"] = (df_prepare["pmer"] / 100)

df_prepare["tendance_pression_hpa"] = (df_prepare["tend"] / 100)

In [ ]:
df_prepare = df_prepare.drop(
    columns=[
        "t",
        "td",
        "pmer",
        "tend"
    ])


In [ ]:
df_prepare["mois"] = (
    df_prepare["validity_time"].dt.month
)

df_prepare["jour"] = (
    df_prepare["validity_time"].dt.day
)

df_prepare["heure"] = (
    df_prepare["validity_time"].dt.hour
)

df_prepare["jour_semaine"] = (
    df_prepare["validity_time"].dt.dayofweek
)

In [ ]:
def determiner_saison(mois):
    if mois in [12, 1, 2]:
        return 0
    elif mois in [3, 4, 5]:
        return 1
    elif mois in [6, 7, 8]:
        return 2
    else:
        return 3

In [ ]:
df_prepare["saison"] = (
    df_prepare["mois"]
    .apply(determiner_saison))

In [ ]:
print("Effectifs :")
print(df_prepare["pluie_prochaine_3h"].value_counts())

print("\nPourcentages :")
print(df_prepare["pluie_prochaine_3h"].value_counts(normalize=True) * 100)

In [ ]:
class_weight="balanced"

In [ ]:
valeurs_manquantes = (
    df_prepare
    .isna()
    .sum()
    .sort_values(ascending=False))

valeurs_manquantes

In [ ]:
pourcentage_manquant = (
    df_prepare
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False))

pourcentage_manquant

In [ ]:
df_prepare = df_prepare.drop(
    columns=["n"])

In [ ]:
print("Dimensions finales :", df_prepare.shape)

print("\nColonnes :")
print(df_prepare.columns.tolist())

print("\nValeurs manquantes restantes :")
print(
    df_prepare
    .isna()
    .sum()
    .sort_values(ascending=False))

print( "\nDoublons :",df_prepare.duplicated().sum())

In [ ]:
df_prepare = (
    df_prepare
    .drop_duplicates()
    .reset_index(drop=True))

In [ ]:
connexion = sqlite3.connect(
    "goutte_eau.db")

df_stations.to_sql(
    name="stations",
    con=connexion,
    if_exists="replace",
    index=False)

df_prepare.to_sql(
    name="observations_meteo",
    con=connexion,
    if_exists="replace",
    index=False)

In [ ]:
requete_tables = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(requete_tables,connexion)

In [ ]:
requete_nombre = """
SELECT COUNT(*) AS nombre_observations
FROM observations_meteo;
"""

pd.read_sql_query(
    requete_nombre,
    connexion)

In [ ]:
requete_stations = """
SELECT DISTINCT
    geo_id_wmo,
    name
FROM observations_meteo
ORDER BY geo_id_wmo;
"""

pd.read_sql_query(
    requete_stations,
    connexion
)

In [ ]:
connexion.close()

In [ ]:
connexion = sqlite3.connect("goutte_eau.db")

df_model = pd.read_sql_query(
    """
    SELECT *
    FROM observations_meteo
    ORDER BY geo_id_wmo, validity_time;
    """, connexion)

connexion.close()

In [ ]:
df_model["validity_time"] = pd.to_datetime(
    df_model["validity_time"],
    errors="coerce",
    utc=True)

df_model = df_model.dropna(
    subset=["validity_time"]).copy()

df_model = df_model.sort_values(
    ["geo_id_wmo", "validity_time"]).reset_index(drop=True)

In [ ]:
features = [
    "lat",
    "lon",
    "dd",
    "ff",
    "u",
    "vv",
    "rr1",
    "temperature_c",
    "point_rosee_c",
    "pression_hpa",
    "tendance_pression_hpa",
    "mois",
    "jour",
    "heure",
    "jour_semaine",
    "saison"
]

target = "pluie_prochaine_3h"

In [ ]:
train_parts = []
test_parts = []

for station_id, groupe in df_model.groupby(
    "geo_id_wmo"
):
    groupe = groupe.sort_values(
        "validity_time"
    ).copy()

    indice_station = int(
        len(groupe) * 0.80
    )

    train_parts.append(
        groupe.iloc[:indice_station]
    )

    test_parts.append(
        groupe.iloc[indice_station:]
    )

In [ ]:
train_parts = []
test_parts = []

for station_id, groupe in df_model.groupby(
    "geo_id_wmo"):

    groupe = groupe.sort_values(
        "validity_time").copy()

    indice_station = int(len(groupe) * 0.80)

    train_parts.append(groupe.iloc[:indice_station])

    test_parts.append(groupe.iloc[indice_station:])

In [ ]:
df_train = (
    pd.concat(train_parts)
    .sort_values(
        ["validity_time", "geo_id_wmo"])
    .reset_index(drop=True))

df_test = (
    pd.concat(test_parts)
    .sort_values(
        ["validity_time", "geo_id_wmo"])
    .reset_index(drop=True))

In [ ]:
X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_test = df_test[features].copy()
y_test = df_test[target].copy()

In [ ]:
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

print("\nRépartition entraînement :")
print(y_train.value_counts(normalize=True) * 100)

print("\nRépartition test :")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
for station_id in sorted(
    df_model["geo_id_wmo"].unique()
):
    train_station = df_train[
        df_train["geo_id_wmo"]
        == station_id
    ]

    test_station = df_test[
        df_test["geo_id_wmo"]
        == station_id
    ]

    print(
        station_id,
        "| fin train :",
        train_station[
            "validity_time"
        ].max(),
        "| début test :",
        test_station[
            "validity_time"
        ].min()
    )

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix)

In [ ]:
modele_reference = DummyClassifier(
    strategy="most_frequent"
)

modele_reference.fit(
    X_train,
    y_train
)

prediction_reference = modele_reference.predict(
    X_test
)

accuracy_reference = accuracy_score(
    y_test,
    prediction_reference
)

precision_reference = precision_score(
    y_test,
    prediction_reference,
    zero_division=0
)

recall_reference = recall_score(
    y_test,
    prediction_reference,
    zero_division=0
)

f1_reference = f1_score(
    y_test,
    prediction_reference,
    zero_division=0
)

print("Accuracy :", accuracy_reference)
print("Precision :", precision_reference)
print("Recall :", recall_reference)
print("F1-score :", f1_reference)

print(
    classification_report(
        y_test,
        prediction_reference,
        zero_division=0
    )
)

matrice_reference = confusion_matrix(
    y_test,
    prediction_reference
)

print(matrice_reference)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
modele_lr = Pipeline([
    (
        "imputation",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "standardisation",
        StandardScaler()
    ),
    (
        "classification",
        LogisticRegression(
            class_weight="balanced",
            random_state=42,
            max_iter=1000
        )
    )
])

In [ ]:
modele_lr.fit(
    X_train,
    y_train)

In [ ]:
prediction_lr = modele_lr.predict(X_test)

In [ ]:
probabilite_lr = (modele_lr.predict_proba(X_test)[:, 1])

In [ ]:
accuracy_lr = accuracy_score(
    y_test,
    prediction_lr)

precision_lr = precision_score(
    y_test,
    prediction_lr,
    zero_division=0)

recall_lr = recall_score(
    y_test,
    prediction_lr,
    zero_division=0)

f1_lr = f1_score(
    y_test,
    prediction_lr,
    zero_division=0)

print("Accuracy :", accuracy_lr)
print("Precision :", precision_lr)
print("Recall :", recall_lr)
print("F1-score :", f1_lr)

print(classification_report(
        y_test,
        prediction_lr,
        zero_division=0))

matrice_lr = confusion_matrix(
    y_test,
    prediction_lr)

print(matrice_lr)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modele_rf = Pipeline([
    (
        "imputation",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "classification",
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42
        )
    )
])

modele_rf.fit(
    X_train,
    y_train
)

prediction_rf = modele_rf.predict(
    X_test
)

probabilite_rf = modele_rf.predict_proba(
    X_test
)[:, 1]

accuracy_rf = accuracy_score(
    y_test,
    prediction_rf
)

precision_rf = precision_score(
    y_test,
    prediction_rf,
    zero_division=0
)

recall_rf = recall_score(
    y_test,
    prediction_rf,
    zero_division=0
)

f1_rf = f1_score(
    y_test,
    prediction_rf,
    zero_division=0
)

print("Accuracy :", accuracy_rf)
print("Precision :", precision_rf)
print("Recall :", recall_rf)
print("F1-score :", f1_rf)

print(
    classification_report(
        y_test,
        prediction_rf,
        zero_division=0
    )
)

matrice_rf = confusion_matrix(
    y_test,
    prediction_rf
)

print(matrice_rf)

In [ ]:
resultats_modeles = pd.DataFrame({
    "Modèle": [
        "DummyClassifier",
        "Régression Logistique",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_reference,
        accuracy_lr,
        accuracy_rf
    ],
    "Precision_pluie": [
        precision_reference,
        precision_lr,
        precision_rf
    ],
    "Recall_pluie": [
        recall_reference,
        recall_lr,
        recall_rf
    ],
    "F1_pluie": [
        f1_reference,
        f1_lr,
        f1_rf
    ]
})

resultats_modeles.round(3)


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    probabilite_lr
)

print("ROC AUC :", auc)

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, seuils = roc_curve(
    y_test,
    probabilite_lr)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    label=f"Régression Logistique (AUC = {auc:.3f})"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    label="Classification aléatoire")

plt.xlabel("Taux de Faux Positifs (False Positive Rate)")
plt.ylabel("Taux de Vrais Positifs (True Positive Rate)")
plt.title("Courbe ROC")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import joblib

joblib.dump(
    modele_lr,
    "modele_pluie.joblib"
)

joblib.dump(
    features,
    "features_modele.joblib")

print("Modèle et liste des variables sauvegardés.")

In [ ]:
import os

print(os.path.exists("modele_pluie.joblib"))
print(os.path.exists("features_modele.joblib"))

In [ ]:
modele_charge = joblib.load("modele_pluie.joblib")

features_chargees = joblib.load(
    "features_modele.joblib"
)

print(features_chargees)

In [ ]:
exemple_test = X_test.head(5)

predictions_test = modele_charge.predict(
    exemple_test
)

probabilites_test = modele_charge.predict_proba(
    exemple_test
)[:, 1]

print("Prédictions :", predictions_test)
print("Probabilités :", probabilites_test)

In [ ]:
fichiers = [
    "modele_pluie.joblib",
    "features_modele.joblib",
    "goutte_eau.db"
]

for fichier in fichiers:
    print(fichier, os.path.exists(fichier))

In [ ]:
%%writefile app.py

from flask import Flask, jsonify, request
import joblib
import pandas as pd
import sqlite3

app = Flask(__name__)

MODELE_PATH = "modele_pluie.joblib"
FEATURES_PATH = "features_modele.joblib"
DATABASE_PATH = "goutte_eau.db"

modele = joblib.load(MODELE_PATH)
features = joblib.load(FEATURES_PATH)


def determiner_saison(mois):
    if mois in [12, 1, 2]:
        return 0
    elif mois in [3, 4, 5]:
        return 1
    elif mois in [6, 7, 8]:
        return 2
    return 3


def recuperer_observation(date_sql, station_id):
    connexion = sqlite3.connect(DATABASE_PATH)

    requete = """
    SELECT
        validity_time,
        geo_id_wmo,
        name,
        lat,
        lon,
        dd,
        ff,
        u,
        vv,
        rr1,
        temperature_c,
        point_rosee_c,
        pression_hpa,
        tendance_pression_hpa
    FROM observations_meteo
    WHERE validity_time = ?
      AND geo_id_wmo = ?
    LIMIT 1
    """

    observation = pd.read_sql_query(
        requete,
        connexion,
        params=(date_sql, station_id)
    )

    connexion.close()
    return observation


@app.route("/", methods=["GET"])
def accueil():
    return jsonify({
        "nom": "API Goutte d'Eau",
        "description": "Estimation du risque de pluie à trois heures",
        "routes": {
            "sante": "GET /health",
            "stations": "GET /stations",
            "prediction": "POST /predict"
        }
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "modele": "Régression Logistique"
    })


@app.route("/stations", methods=["GET"])
def stations():
    connexion = sqlite3.connect(DATABASE_PATH)

    resultat = pd.read_sql_query(
        """
        SELECT DISTINCT
            geo_id_wmo AS station_id,
            name AS station
        FROM observations_meteo
        ORDER BY geo_id_wmo
        """,
        connexion
    )

    connexion.close()

    return jsonify(
        resultat.to_dict(orient="records")
    )


@app.route("/predict", methods=["POST"])
def predict():
    contenu = request.get_json(silent=True)

    if contenu is None:
        return jsonify({
            "erreur": "Le corps doit être au format JSON."
        }), 400

    date_demandee = contenu.get("date")
    station_id = contenu.get("station_id")

    if not date_demandee or station_id is None:
        return jsonify({
            "erreur": "Les champs date et station_id sont obligatoires."
        }), 400

    try:
        station_id = int(station_id)
        date_convertie = pd.to_datetime(
            date_demandee,
            utc=True
        )
    except (TypeError, ValueError):
        return jsonify({
            "erreur": "Date ou identifiant de station invalide."
        }), 400

    date_sql = date_convertie.isoformat(sep=" ")

    observation = recuperer_observation(
        date_sql,
        station_id
    )

    if observation.empty:
        return jsonify({
            "erreur": "Aucune observation trouvée pour cette date et cette station."
        }), 404

    observation["validity_time"] = pd.to_datetime(
        observation["validity_time"],
        utc=True
    )

    observation["mois"] = observation["validity_time"].dt.month
    observation["jour"] = observation["validity_time"].dt.day
    observation["heure"] = observation["validity_time"].dt.hour
    observation["jour_semaine"] = observation["validity_time"].dt.dayofweek
    observation["saison"] = observation["mois"].apply(
        determiner_saison
    )

    X_prediction = observation[features].copy()

    classe = int(
        modele.predict(X_prediction)[0]
    )

    probabilite = float(
        modele.predict_proba(X_prediction)[0, 1]
    )

    if probabilite < 0.30:
        niveau = "faible"
    elif probabilite < 0.60:
        niveau = "modéré"
    else:
        niveau = "élevé"

    return jsonify({
        "date_observation": str(date_demandee),
        "station_id": station_id,
        "station": observation["name"].iloc[0],
        "horizon_prediction": "3 heures",
        "classe_predite": classe,
        "pluie_predite": bool(classe),
        "probabilite_pluie": round(probabilite, 4),
        "probabilite_pluie_pourcentage": round(
            probabilite * 100,
            2
        ),
        "niveau_risque": niveau
    })


if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )

In [ ]:
from app import app

client = app.test_client()

In [ ]:
reponse = client.get("/health")

print("Code HTTP :", reponse.status_code)
print(reponse.get_json())

In [ ]:
reponse = client.get("/stations")

print("Code HTTP :", reponse.status_code)
print(reponse.get_json())

In [ ]:
connexion = sqlite3.connect("goutte_eau.db")

exemple_base = pd.read_sql_query(
    """
    SELECT
        validity_time,
        geo_id_wmo,
        name
    FROM observations_meteo
    LIMIT 1
    """,
    connexion
)

connexion.close()

exemple_base

In [ ]:
date_test = exemple_base.iloc[0]["validity_time"]
station_test = int(
    exemple_base.iloc[0]["geo_id_wmo"]
)

print(date_test)
print(station_test)

In [ ]:
reponse = client.post(
    "/predict",
    json={
        "date": str(date_test),
        "station_id": station_test
    }
)

print("Code HTTP :", reponse.status_code)
print(reponse.get_json())

In [ ]:
reponse = client.post(
    "/predict",
    json={
        "date": "2030-01-01T00:00:00Z",
        "station_id": 7535
    }
)

print(reponse.status_code)
print(reponse.get_json())

In [ ]:
reponse = client.post(
    "/predict",
    json={
        "station_id": 7535
    }
)

print(reponse.status_code)
print(reponse.get_json())

In [ ]:
%%writefile streamlit_app.py